# 08 - Master Comparison

The headline table of the portfolio. This notebook does
not train anything. It reads each model's
results/<slug>/metrics.json, builds the side-by-side
pandas summary, and shows the cross-model verdict (does
the non-linear model break the linear ceiling or not?).

Run this AFTER the seven model notebooks (or the seven
train_*.py scripts) finished and filled results/.

## 1 - Setup

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
from IPython.display import Image, display

from src.train_utils import RESULTS_DIR
from evaluate_all_results import (
    load_metrics_files,
    order_payloads,
    missing_from_preferred,
    row_from_payload,
    build_summary_dataframe,
    PREFERRED_ORDER,
)

print(f"RESULTS_DIR = {RESULTS_DIR}")

## 2 - Discover what's inside results/

Here we list every model sub-directory under results/ and
check that the canonical 7-model lineup is all there.

In [ ]:
payloads = order_payloads(load_metrics_files(RESULTS_DIR))
missing = missing_from_preferred(payloads)

print(f"Found {len(payloads)} model result file(s).")
if missing:
    print(f"Missing from the expected lineup: {missing}")
else:
    print("All 7 expected models present.")

## 3 - The headline summary table

Each row is one model, and the columns are accuracy and
F1 on Baseline and on Advanced, plus the Delta between
them. A positive Delta means the engineered culinary
features helped that model.

In [ ]:
df = build_summary_dataframe(RESULTS_DIR)

def _style_deltas(v):
    if not isinstance(v, (int, float)) or pd.isna(v):
        return ""
    if v > 0.001:
        return "color: green"
    if v < -0.001:
        return "color: crimson"
    return "color: dimgray"

(
    df.style
      .format({
          "Acc (Baseline)": "{:.4f}",
          "Acc (Advanced)": "{:.4f}",
          "Δ Acc": "{:+.4f}",
          "F1 (Baseline)": "{:.4f}",
          "F1 (Advanced)": "{:.4f}",
          "Δ F1": "{:+.4f}",
      })
      .map(_style_deltas, subset=["Δ Acc", "Δ F1"])
      .set_caption("7-model A/B comparison: Baseline vs Advanced (engineered culinary features)")
)

## 4 - Cross-model verdict (linear baseline vs non-linear)

LR is the calibrated linear baseline. Random Forest and
MLP are the non-linear contenders. The "meaningful gain"
we set is 1 percentage point on either Acc or F1.

In [ ]:
MEANINGFUL = 0.01
by_name = {p["model_name"]: p for p in payloads}
lr  = by_name.get("LogisticRegression")
rf  = by_name.get("RandomForest")
mlp = by_name.get("MLP (128,64)")
assert lr is not None, "LogisticRegression metrics missing"

def _classify(model_name, p):
    lr_a = lr["datasets"]["Advanced"]
    ad = p["datasets"]["Advanced"]
    d_acc = ad["accuracy"] - lr_a["accuracy"]
    d_f1 = ad["f1"] - lr_a["f1"]
    if d_acc >= MEANINGFUL and d_f1 >= MEANINGFUL:
        verdict = "BREAKS the linear ceiling on both metrics"
    elif d_acc >= MEANINGFUL or d_f1 >= MEANINGFUL:
        verdict = "PARTIAL gain (one metric only)"
    elif d_acc <= -MEANINGFUL or d_f1 <= -MEANINGFUL:
        verdict = "UNDERPERFORMS the linear baseline"
    else:
        verdict = "PLATEAUS within noise of LR"
    return {
        "Model": model_name,
        "Acc (Advanced)": ad["accuracy"],
        "Δ vs LR (Acc)": d_acc,
        "F1 (Advanced)": ad["f1"],
        "Δ vs LR (F1)": d_f1,
        "Verdict": verdict,
    }

verdict_rows = []
if rf is not None: verdict_rows.append(_classify("RandomForest", rf))
if mlp is not None: verdict_rows.append(_classify("MLP (128,64)", mlp))
verdict_df = pd.DataFrame(verdict_rows).set_index("Model")

display(verdict_df.style.format({
    "Acc (Advanced)": "{:.4f}",
    "F1 (Advanced)": "{:.4f}",
    "Δ vs LR (Acc)": "{:+.4f}",
    "Δ vs LR (F1)": "{:+.4f}",
}))

## 5 - The PCA + KNN before/after story

This is the most diagnostic-driven result in the project:
dropping the four nutrition columns before PCA grows the
retained search space from 1 component to about 200, and
lifts KNN accuracy by about 5 percentage points. README
section 4 Lesson #5.

In [ ]:
orig = by_name.get("PCA(0.90) + KNN")
improved = by_name.get("PCA(0.90) + KNN (Improved)")

if orig is not None and improved is not None:
    pca_orig = orig["extras"]["pca_components_retained"]
    pca_imp  = improved["extras"]["pca_components_retained"]

    cmp = pd.DataFrame({
        "Original - components": pca_orig,
        "Improved - components": pca_imp,
        "Original - Acc": {ds: orig["datasets"][ds]["accuracy"] for ds in pca_orig},
        "Improved - Acc": {ds: improved["datasets"][ds]["accuracy"] for ds in pca_imp},
        "Δ Acc": {ds: improved["datasets"][ds]["accuracy"] - orig["datasets"][ds]["accuracy"] for ds in pca_imp},
    })
    display(cmp.style.format({
        "Original - Acc": "{:.4f}",
        "Improved - Acc": "{:.4f}",
        "Δ Acc": "{:+.4f}",
    }))

## 6 - Embedded plot gallery

The headline plot of each model, side by side. The images
come from results/<slug>/ so they show whatever was last
written by the model's notebook (or by the matching
train_<model>.py script).

In [ ]:
plot_gallery = [
    ("Logistic Regression - Confusion Matrix", "logistic_regression/confusion_matrix.png"),
    ("Logistic Regression - ROC Curve",        "logistic_regression/roc_curve.png"),
    ("Random Forest - Feature Importance",     "random_forest/feature_importance.png"),
    ("MLP - Loss Curve + Validation Overlay",  "mlp/loss_curve.png"),
]

for caption, rel_path in plot_gallery:
    full_path = RESULTS_DIR / rel_path
    if full_path.exists():
        print(caption)
        display(Image(str(full_path)))
    else:
        print(f"[missing] {caption} ({rel_path})")

## 7 - Final verdict (one-liner per model class)

- **Linear interpretable champion:** Logistic Regression
  (calibrated probabilities, signed coefficients).
- **Predictive champion:** Random Forest - it breaks the
  linear ceiling by +2.28 pp Acc / +3.49 pp F1.
- **MLP:** clears the linear ceiling too once we add early
  stopping (+1.15 pp Acc / +2.27 pp F1 vs LR) - but only
  then; unregularised it lands far below LR.
- **KNN (cautionary tale -> algorithmic fix):** the
  diagnostic-driven Improved variant lifts accuracy by
  +4.5 to +6.3 pp just by dropping the outlier-dominated
  columns before PCA.

For the full narrative see results/research.md.